# Find Organization Axes

Discover what dimensions work best for organizing your videos and images.
Let Jockey recommend the most useful categorization strategies, scored by how cleanly they separate your content.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import json
import os

from twelvelabs import TwelveLabs, TextParam
from twelvelabs.types.text_param_format import TextParamFormat_JsonSchema

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "<YOUR_API_KEY>")
STORE_ID = os.environ.get("TWELVELABS_STORE_ID", "<YOUR_KNOWLEDGE_STORE_ID>")  # Replace with your knowledge store ID

client = TwelveLabs(api_key=API_KEY)

## Helper Functions

A utility to extract text content from a Jockey API response.

In [ ]:
def parse_response(response) -> str:
    """Extract text content from a Jockey response.

    Args:
        response: The ResponseObject returned by client.responses.create().

    Returns:
        The text content from the first message output, or an empty string
        if no message content is found.
    """
    for output in response.output:
        if output.type == "message":
            for content in output.content:
                return content.text
    return ""

## Axes Schema

Define a JSON schema for organization axis recommendations. Each axis includes a name,
description, expected number of groups, example categories, an effectiveness score (0-1),
and a rationale explaining why it works well for this collection.

In [ ]:
AXES_SCHEMA = {
    "type": "object",
    "properties": {
        "axes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "axis": {"type": "string"},
                    "description": {"type": "string"},
                    "expected_groups": {"type": "integer"},
                    "example_categories": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "effectiveness_score": {"type": "number"},
                    "rationale": {"type": "string"},
                },
            },
        },
        "recommendation": {"type": "string"},
    },
}

## Find the Best Organization Axes

Use the `instructions` field to set Jockey's role as a content strategist, then ask it to
analyze the collection and recommend the top 5 organization dimensions. Each axis is scored
from 0 to 1 based on how cleanly it separates the content into distinct, useful groups.

In [ ]:
response = client.responses.create(
    knowledge_store_id=STORE_ID,
    instructions=(
        "You are a content strategist. Analyze these videos and images and "
        "recommend the most effective ways to organize it. Score each axis "
        "by how well it separates content into distinct, useful groups."
    ),
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "What are the top 5 best ways to organize these videos and images? "
                "Score each from 0-1 based on how cleanly it separates the content."
            ),
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="organization_axes", schema_=AXES_SCHEMA)
    ),
)

data = json.loads(parse_response(response))

print(f"Recommendation: {data['recommendation']}\n")
for ax in data["axes"]:
    print(f"  {ax['axis']} (score: {ax['effectiveness_score']})")
    print(f"    {ax['description']}")
    print(f"    Groups: {ax['expected_groups']} -- {', '.join(ax['example_categories'])}")
    print(f"    Why: {ax['rationale']}\n")

## Variations

Modify the prompt or instructions to tailor the recommendations:

- **Domain-constrained:** "What are the best ways to organize these for a marketing team?"
- **User-centric:** "How would different audiences want this organized?"
- **Hierarchical:** "Suggest a two-level taxonomy: primary axis and sub-axes"

## Next Steps

- **[Get Corpus Overview](get_corpus_overview.ipynb)** -- understand the full collection first
- **[Search Videos](search_videos.ipynb)** -- find specific moments by description
- **[Extract Entities](extract_entities.ipynb)** -- list all people, places, objects, and concepts
- **[Enrich Content](enrich_content.ipynb)** -- get deeper, domain-specific metadata

See also:
- [Structured Output Guide](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-response/structured-output) -- more on JSON schema responses